In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
from scipy import stats

session = get_active_session()
session.sql("USE DATABASE NHANES_DB").collect()
session.sql("USE SCHEMA RAW").collect()

df = session.table("NHANES_FINAL").to_pandas()

# Same cleaning as before
df_clean = df.dropna(subset=['SYSTOLIC_BP', 'DIASTOLIC_BP'])

diet_cols = ['SODIUM_MG','FIBER_G','SUGAR_G','CALORIES','PROTEIN_G','FAT_G']
df_clean[diet_cols] = df_clean[diet_cols].fillna(df_clean[diet_cols].median())
df_clean['BMI'] = df_clean['BMI'].fillna(df_clean['BMI'].median())

print("Ready! Shape:", df_clean.shape)

In [ ]:
# 1. Sodium vs Hypertension
t, p1 = stats.ttest_ind(
    df_clean[df_clean['HYPERTENSION_FLAG'] == 1]['SODIUM_MG'],
    df_clean[df_clean['HYPERTENSION_FLAG'] == 0]['SODIUM_MG']
)
print(f"1. Sodium vs Hypertension:       p={p1:.4f}", "✅" if p1 < 0.05 else "❌")

In [ ]:
# 2. Sodium vs BMI
t, p2 = stats.ttest_ind(
    df_clean[df_clean['SODIUM_MG'] > 2300]['BMI'],
    df_clean[df_clean['SODIUM_MG'] <= 2300]['BMI']
)
print(f"2. Sodium vs BMI:                p={p2:.4f}", "✅" if p2 < 0.05 else "❌")

In [ ]:
# 3. BMI across Age Groups
df_clean['AGE_GROUP'] = pd.cut(df_clean['AGE'], bins=[18,30,40,50,60,100],
                                labels=['18-30','31-40','41-50','51-60','60+'])
f, p3 = stats.f_oneway(*[g['BMI'].values for _, g in df_clean.groupby('AGE_GROUP')])
print(f"3. BMI across Age Groups:        p={p3:.4f}", "✅" if p3 < 0.05 else "❌")

In [ ]:
# 4. High Sodium vs Hypertension
chi2, p4, _, _ = stats.chi2_contingency(
    pd.crosstab(df_clean['SODIUM_MG'] > 2300, df_clean['HYPERTENSION_FLAG']))
print(f"4. High Sodium vs Hypertension:  p={p4:.4f}", "✅" if p4 < 0.05 else "❌")

In [ ]:
# 5. Fiber vs Obesity
t, p5 = stats.ttest_ind(
    df_clean[df_clean['BMI_CATEGORY'] == 'Obese']['FIBER_G'],
    df_clean[df_clean['BMI_CATEGORY'] != 'Obese']['FIBER_G']
)
print(f"5. Fiber vs Obesity:             p={p5:.4f}", "✅" if p5 < 0.05 else "❌")